# Performance Innovative Task: Traveling Salesperson Problem (TSP)
## Optimization of Last-Mile Delivery Routing in Cagayan de Oro City Using Particle Swarm Optimization (PSO)

**Course:** Graph Theory (3rd Year)  
**Assigned Algorithm:** Section 2.4 — Particle Swarm Optimization (PSO)  
**Primary Reference:** Luna-Garcia, R., Candano, T. M. C., & Caga-anan, R. L. (2024). *Performance Analysis of Classical Algorithms for the Traveling Salesman Problem*. The Mindanawan Journal of Mathematics (TMJM), MSU-IIT, Vol. 6, No. 2, pp. 79–111. [https://doi.org/10.62071/tmjm.v6i2.716](https://doi.org/10.62071/tmjm.v6i2.716)

### 1. Problem Setup: Cagayan de Oro City Locations & Road Distance Matrix

In [ ]:
import math
import random
import itertools
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# 8 Strategic Hubs in Cagayan de Oro City
LOCATIONS = [
    {"id": 0, "code": "V0", "name": "Plaza Divisoria (Central Hub)", "lat": 8.47775, "lon": 124.64378, "role": "Depot"},
    {"id": 1, "code": "V1", "name": "Ayala Malls Centrio",           "lat": 8.48510, "lon": 124.65120, "role": "Commercial"},
    {"id": 2, "code": "V2", "name": "SM CDO Downtown Premier",       "lat": 8.48440, "lon": 124.65430, "role": "Commercial"},
    {"id": 3, "code": "V3", "name": "Limketkai Center",              "lat": 8.48160, "lon": 124.65390, "role": "Commercial"},
    {"id": 4, "code": "V4", "name": "Carmen Commercial Center",      "lat": 8.48090, "lon": 124.63550, "role": "Commercial"},
    {"id": 5, "code": "V5", "name": "Bulua Westbound Terminal",      "lat": 8.50430, "lon": 124.61590, "role": "Transit/Logistics"},
    {"id": 6, "code": "V6", "name": "Port of CDO (Macabalan)",       "lat": 8.49390, "lon": 124.66230, "role": "Seaport"},
    {"id": 7, "code": "V7", "name": "SM City CDO Uptown",             "lat": 8.45410, "lon": 124.62300, "role": "Commercial"}
]

LOCATION_NAMES = [loc["name"] for loc in LOCATIONS]
LOCATION_CODES = [loc["code"] for loc in LOCATIONS]

# Pairwise Google Maps actual road driving distance matrix (in km)
DIST_MATRIX = [
    [0.0, 1.2, 1.6, 1.8, 1.3, 5.2, 3.8, 6.5],  # V0: Divisoria
    [1.2, 0.0, 0.4, 1.1, 2.4, 5.6, 3.1, 7.2],  # V1: Centrio
    [1.6, 0.4, 0.0, 0.8, 2.6, 6.0, 2.7, 7.5],  # V2: SM Downtown
    [1.8, 1.1, 0.8, 0.0, 2.8, 6.5, 3.3, 7.8],  # V3: Limketkai
    [1.3, 2.4, 2.6, 2.8, 0.0, 4.5, 4.6, 5.8],  # V4: Carmen
    [5.2, 5.6, 6.0, 6.5, 4.5, 0.0, 6.8, 9.2],  # V5: Bulua
    [3.8, 3.1, 2.7, 3.3, 4.6, 6.8, 0.0, 9.8],  # V6: Macabalan
    [6.5, 7.2, 7.5, 7.8, 5.8, 9.2, 9.8, 0.0]   # V7: SM Uptown
]

for loc in LOCATIONS:
    print(f"{loc['code']}: {loc['name']} | ({loc['lat']}°N, {loc['lon']}°E) | Type: {loc['role']}")

### 2. Particle Swarm Optimization (PSO) Algorithm for Discrete TSP
Following Section 2.4 and Appendix 4.4 from Luna-Garcia et al. (2024).

In [ ]:
def calculate_tour_distance(tour, matrix=DIST_MATRIX):
    total = 0.0
    n = len(tour)
    for i in range(n - 1):
        total += matrix[tour[i]][tour[i + 1]]
    total += matrix[tour[-1]][tour[0]]
    return round(total, 4)

class Particle:
    def __init__(self, num_cities):
        self.position = random.sample(range(num_cities), num_cities)
        self.velocity = []
        self.best_position = list(self.position)
        self.best_cost = float('inf')
        self.current_cost = float('inf')

def pso_tsp(matrix=DIST_MATRIX, num_particles=40, max_iterations=120, w=0.5, c1=1.5, c2=1.5, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    num_cities = len(matrix)
    particles = [Particle(num_cities) for _ in range(num_particles)]
    global_best_position = None
    global_best_cost = float('inf')
    convergence = []

    for iteration in range(max_iterations):
        for particle in particles:
            particle.current_cost = calculate_tour_distance(particle.position, matrix)
            if particle.current_cost < particle.best_cost:
                particle.best_cost = particle.current_cost
                particle.best_position = list(particle.position)
            if particle.current_cost < global_best_cost:
                global_best_cost = particle.current_cost
                global_best_position = list(particle.position)
        convergence.append(global_best_cost)

        for particle in particles:
            new_velocity = []
            # Inertia component: random swaps
            for _ in range(random.randint(1, max(1, num_cities // 2))):
                if random.random() < w:
                    idx1, idx2 = random.sample(range(num_cities), 2)
                    new_velocity.append((idx1, idx2))
            # Cognitive component: personal best
            for i in range(num_cities):
                if random.random() < c1:
                    if particle.position[i] != particle.best_position[i]:
                        idx1 = particle.position.index(particle.best_position[i])
                        new_velocity.append((idx1, i))
            # Social component: global best
            for i in range(num_cities):
                if random.random() < c2:
                    if particle.position[i] != global_best_position[i]:
                        idx1 = particle.position.index(global_best_position[i])
                        new_velocity.append((idx1, i))
            for idx1, idx2 in new_velocity:
                particle.position[idx1], particle.position[idx2] = particle.position[idx2], particle.position[idx1]
            particle.velocity = new_velocity

    return global_best_position, global_best_cost, convergence

print('PSO Solver defined successfully.')

### 3. Execution, Benchmark Comparison, and Solution Audit

In [ ]:
# Ground Truth Exact Benchmark: Brute Force O(n!)
cities = list(range(1, len(DIST_MATRIX)))
min_dist = float('inf')
bf_tour = None
for p in itertools.permutations(cities):
    tour = [0] + list(p)
    d = calculate_tour_distance(tour)
    if d < min_dist:
        min_dist = d
        bf_tour = tour
print(f"Brute Force Exact Optimal Distance: {min_dist:.2f} km")

# Run PSO Optimization
pso_best, pso_cost, convergence = pso_tsp(DIST_MATRIX, num_particles=40, max_iterations=120, seed=123)
idx0 = pso_best.index(0)
opt_tour = pso_best[idx0:] + pso_best[:idx0]

print(f"PSO Discovered Distance:          {pso_cost:.2f} km")
print(f"Optimality Status:                 {'100% PROVEN GLOBAL OPTIMUM!' if abs(pso_cost - min_dist) < 1e-4 else 'Approximate Solution'}")
tour_str = ' -> '.join([LOCATION_CODES[i] for i in opt_tour] + [LOCATION_CODES[opt_tour[0]]])
print(f"Optimized Closed Tour:             {tour_str}")

print('\n' + '=' * 75)
print(f"{'Leg':<5} {'From':<28} {'To':<28} {'Distance (km)':>12}")
print('-' * 75)
total_recalc = 0.0
for i in range(len(opt_tour)):
    u = opt_tour[i]
    v = opt_tour[(i + 1) % len(opt_tour)]
    leg_d = DIST_MATRIX[u][v]
    total_recalc += leg_d
    from_name = f"{LOCATION_CODES[u]}: {LOCATION_NAMES[u]}"
    to_name = f"{LOCATION_CODES[v]}: {LOCATION_NAMES[v]}"
    print(f"{i+1:<5} {from_name:<28} {to_name:<28} {leg_d:>12.2f}")
print('-' * 75)
print(f"{'TOTAL RECALCULATED DISTANCE:':<62} {total_recalc:>12.2f} km")
print('=' * 75)